In [ ]:
# --- Data Preprocessing ---
# 1. Data preprocessing and coarse-graining -> see sec2_data

# --- Modeling ---
# Symbolic regression with PySR or GP-GOMEA
from pysr import PySRRegressor
import xarray as xr
import numpy as np

import psutil
import os

T_steps = 50  # number of time steps to consider (to confine memory usage)

# Load preprocessed data
# Example: variables: temperature, relative_humidity, pressure, cloud_cover
# By coords sorts the time dimension correctly
clc = xr.open_mfdataset('/home/b/b309170/bd1179_work/DYAMOND/hvcg_data/clc/*', combine='by_coords').clc[:T_steps, 4:-1].values

# For relative humidity
pa = xr.open_mfdataset('/home/b/b309170/bd1179_work/DYAMOND/hvcg_data/pa/*', combine='by_coords').pa[:T_steps].values
ta = xr.open_mfdataset('/home/b/b309170/bd1179_work/DYAMOND/hvcg_data/ta/*', combine='by_coords').ta[:T_steps].values
hus = xr.open_mfdataset('/home/b/b309170/bd1179_work/DYAMOND/hvcg_data/hus/*', combine='by_coords').hus[:T_steps].values

zg = xr.open_mfdataset('/home/b/b309170/bd1179_work/DYAMOND/hvcg_data/zg/*', combine='by_coords').zg[:T_steps].values
zg = np.repeat(np.expand_dims(zg, axis=0), repeats=T_steps, axis=0)

T0 = 273.15
r = 0.00263*pa*hus*np.exp((17.67*(ta-T0))/(ta-29.65))**(-1) 
dzrh = (r[:, 1:] - r[:, :-1])/(zg[:, 1:] - zg[:, :-1])

# Remaining input features
clw = xr.open_mfdataset('/home/b/b309170/bd1179_work/DYAMOND/hvcg_data/clw/*', combine='by_coords').clw[:T_steps, 4:-1].values
cli = xr.open_mfdataset('/home/b/b309170/bd1179_work/DYAMOND/hvcg_data/cli/*', combine='by_coords').cli[:T_steps, 4:-1].values
ta = ta[:, 4:-1]
r = r[:, 4:-1]
dzrh = dzrh[:, 3:-1]

# Stack features
X = np.column_stack([np.reshape(r, -1), np.reshape(ta, -1), np.reshape(dzrh, -1), np.reshape(clw, -1), np.reshape(cli, -1)])

# Check memory usage
process = psutil.Process(os.getpid())
mem_info = process.memory_info()
print(f"RSS: {mem_info.rss / 1024 ** 2:.2f} MB")  # Resident memory
print(f"VMS: {mem_info.vms / 1024 ** 2:.2f} MB")  # Virtual memory

# Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, np.reshape(clc, -1), test_size=0.2, random_state=42)

inds = np.random.randint(X_train.shape[0], size=1000)  # Limit to 5000 samples for training

# Train model
model = PySRRegressor(
    niterations=1000,
    unary_operators=["exp", "log", "sqrt"],
    binary_operators=["+", "-", "*", "/"],
    loss="loss(x, y) = (x - y)^2",
    procs=32  # parallelism for cluster
)
model.fit(X_train[inds], y_train[inds]) # Limited to 1000

# --- Evaluation ---
from sklearn.metrics import r2_score
preds = model.predict(X_test)
print("R²:", r2_score(y_test, preds))
print("Discovered equation:", model.get_best())

# Save results
import joblib
joblib.dump(model, "cloudcover_model.pkl")

# --- Visualization ---
import matplotlib.pyplot as plt
plt.scatter(y_test, preds, alpha=0.2)
plt.xlabel("True Cloud Cover")
plt.ylabel("Predicted")
plt.title("Symbolic Regression Performance")
plt.show()

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
RSS: 12777.75 MB
VMS: 81119.93 MB


/home/b/b309170/my_work/Miniconda3/envs/pysr/lib/python3.10/site-packages/pysr/sr.py:1036: FutureWarning: `loss` has been renamed to `elementwise_loss` in PySRRegressor. Please use that instead.
  warnings.warn(
/home/b/b309170/my_work/Miniconda3/envs/pysr/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 2.520e+03
Progress: 15 / 31000 total iterations (0.048%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
2           1.548e-01  0.000e+00  y = log(1.5392)
3           6.715e-02  8.352e-01  y = sqrt(sqrt(x₃))
4           4.609e-02  3.763e-01  y = sqrt(sqrt(sqrt(x₃)))
5           4.244e-02  8.249e-02  y = sqrt(sqrt(x₁ * x₃))
6           3.294e-02  2.535e-01  y = sqrt(sqrt(sqrt(x₄ + x₃)))
7           2.962e-02  1.060e-01  y = sqrt(sqrt(sqrt(sqrt(x₄) + x₃)))
9           2.885e-02  1.316e-02  y = sqrt(sqrt(sqrt((sqrt(x₄) + x₃) + x₃)))
11          2.047e-02  1.716e-01  y = sqrt(sqrt(sqrt(x₄) + sqrt(x₃))) * (x₀ / 0.55283)
13          2.045e-02  5.531e-04  y = sqrt(sqrt(sqrt(x₄) + (x₃ + sqrt(x₃)))) * (x₀ / 0.55283...
                                      )
15        